# Midterm Part A - thin Colab launcher
Sparse-checks-out only `midterm_a/`, installs the project and runs Part A through its local `train.py` entry point. All task code, configs and launch logic stay inside this folder. All logic lives in the repo. Runtime -> Change runtime type -> **T4 GPU**. After a disconnect re-run the setup cell and the same command: runs resume from `last.ckpt`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

os.chdir("/content")

print("Current directory:", os.getcwd())

In [ ]:
FORK_URL = "https://github.com/rafihi/representation-learning-course.git"
WORK = "/content/drive/MyDrive/ssl_midterm_a"      # checkpoints, results and the dataset cache live on Drive
!mkdir -p {WORK}/runs {WORK}/results {WORK}/data_cache
!rm -rf /content/repo && git clone --filter=blob:none --sparse {FORK_URL} /content/repo
%cd /content/repo
!git sparse-checkout set midterm_a
%cd midterm_a
!ln -s {WORK}/runs runs && ln -s {WORK}/results results && ln -s {WORK}/data_cache data_cache
!pip install -q -r requirements.txt

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 0. Sanity checks (no downloads)

In [ ]:
!SSL_FAKE_DATA=1 python -m pytest -q tests

In [ ]:
!set -o pipefail; python train.py --config configs/a1_tune_dae.yaml --smoke --set out_dir=/tmp/smoke_a1 trainer.progress=false 2>&1 | tail -5

## 1. A2 - tuning (proxy) + reconstruction-vs-kNN figure

In [ ]:
from pathlib import Path

out = Path("results/a1_tune_dae")

done = sorted(out.glob("trial_*/metrics.json"))

print(f"Successfully completed: {len(done)} / 20")

for p in sorted(out.glob("trial_*")):
    status = "DONE" if (p / "metrics.json").exists() else "INCOMPLETE"
    checkpoint = (p / "ckpt" / "last.ckpt").exists()
    print(p.name, status, "| Checkpoint:", checkpoint)

print("\nFinal plot exists:",
      (out / "recon_vs_knn.png").exists())

In [ ]:
!python train.py --config configs/a1_tune_dae.yaml

Open `results/a1_tune_dae/results.csv`, pick the trial with the best `knn_top1`, copy its `model.kind / level / lr / weight_decay` into `configs/a2_dae_final.yaml` (commit + push, re-run the setup cell), then:

In [ ]:
import pandas as pd

df = pd.read_csv("results/a1_tune_dae/results.csv")

best = df.loc[df["knn_top1"].idxmax()]

print("BEST TRIAL PARAMETERS")
for col in [
    "trial",
    "model.kind",
    "model.level",
    "model.lr",
    "model.weight_decay",
    "knn_top1",
    "knn_top5"
]:
    print(f"{col}: {best[col]}")

In [ ]:
!grep "model:" configs/a2_dae_final.yaml

In [ ]:
!SSL_FAKE_DATA=1 SSL_TINY_BACKBONE=1 python train.py --config configs/a2_dae_final.yaml --smoke --set out_dir=runs/ckpt_smoke

In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/ssl_midterm_a/runs/ckpt_smoke/ckpt/last.ckpt")

print("Checkpoint exists:", p.exists())

if p.exists():
    print("Size:", round(p.stat().st_size / 1024**2, 2), "MB")

In [ ]:
!python train.py --config configs/a2_dae_final.yaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

root = Path('/content/drive/MyDrive/ssl_midterm_a')

print("=== A2 CHECKPOINT SEARCH ===")

for p in root.rglob("last.ckpt"):
    print("FOUND:", p)
    print("SIZE:", round(p.stat().st_size / 1024**2, 2), "MB")

In [ ]:
from pathlib import Path
import os

paths = [
    Path("/content/repo/midterm_a/runs"),
    Path("/content/repo/midterm_a/runs/dae_final"),
    Path("/content/drive/MyDrive/ssl_midterm_a/runs"),
    Path("/content/drive/MyDrive/ssl_midterm_a/runs/dae_final"),
    Path("/content/drive/MyDrive/ssl_midterm_a/runs/dae_final/ckpt")
]

for p in paths:
    print("\n", p)
    print("Exists:", p.exists())
    print("Is symlink:", p.is_symlink())

    if p.is_symlink():
        print("Target:", os.readlink(p))

    if p.exists() and p.is_dir():
        print("Contents:", [x.name for x in p.iterdir()])

In [ ]:
from pathlib import Path
import os
import yaml

os.chdir("/content/repo/midterm_a")

root = Path("/content/drive/MyDrive/ssl_midterm_a")

# 1. חיפוש כל קובצי המודל ב-Drive
print("=== ALL CHECKPOINTS ===")

for p in root.rglob("*.ckpt"):
    print(p, round(p.stat().st_size / 1024**2, 2), "MB")

# 2. בדיקת הגדרות A2
with open("configs/a2_dae_final.yaml") as f:
    a2 = yaml.safe_load(f)

with open("configs/base_dae.yaml") as f:
    base = yaml.safe_load(f)

print("\n=== A2 CONFIG ===")
print("Output:", a2["out_dir"])
print("Max epochs:", a2["trainer"]["max_epochs"])
print("Resume:", base["resume"])
print("Checkpoint frequency:",
      base["trainer"]["ckpt_every"])

# 3. בדיקת תוכן תיקיית A2
out = root / "runs/dae_final"

print("\n=== A2 DRIVE CONTENTS ===")
for p in out.rglob("*"):
    print(p)

In [ ]:
%cd /content/repo/midterm_a
!ls -ld runs results data_cache
!ls -la runs/dae_final runs/dae_final/ckpt
!ls -la /content/drive/MyDrive/ssl_midterm_a/runs/dae_final

In [ ]:
!python train.py --config configs/a3_vanilla_ae_final.yaml

## 2. A3 - frozen protocol vs. baselines (k-NN + linear probe)

In [ ]:
!python train.py --config configs/a4_eval.yaml

## 3. A3/A4 - fine-tuning and label efficiency

In [ ]:
!python train.py --config configs/a5_finetune.yaml

## 4. A4 - corruption curves

In [ ]:
!python train.py --config configs/a6_robustness.yaml

## 5. A5 - t-SNE, retrieval grid, per-class accuracy

In [ ]:
!python train.py --config configs/a7_interpret.yaml

## Final numbers on the TEST split (only once, after every decision is made)

In [ ]:
!python train.py --config configs/a4_eval_test.yaml
!python train.py --config configs/a5_finetune_test.yaml
!python train.py --config configs/a6_robustness_test.yaml
!python train.py --config configs/a7_interpret_test.yaml

## TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs